In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

# spark = SparkSession.builder \
#     .appName("SparkExample") \
#     .master("local[*]") \
#     .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
#     .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
#     .config('spark.executor.memory', '8g') \
#     .config('spark.driver.memory', '8g') \
#     .getOrCreate()

In [ ]:
qqqq

## actualizar retiro telef 

In [2]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT  NUMERO_DOCUMENTO,cl_telf1, cl_telf2, cl_telf3, cl_telf4, cl_telf5, cl_telf6, cl_telf7, cl_telf8, cl_telf9, cl_telf10, cl_movil, cl_celular, cl_telefono 
FROM crm_target.alfcc_clientes
WHERE cl_base = 'mayo 2026'
and cl_estado=1
"""

df_dni = pd.read_sql(query, engine_mysql)

cols_tel = [
    'cl_telf1','cl_telf2','cl_telf3','cl_telf4','cl_telf5',
    'cl_telf6','cl_telf7','cl_telf8','cl_telf9','cl_telf10',
    'cl_movil','cl_celular','cl_telefono'
]

df_long = df_dni.melt(
    id_vars='NUMERO_DOCUMENTO',
    value_vars=cols_tel,
    var_name='tipo_telf',
    value_name='CELULAR'
)
df_long['CELULAR'] = (
    df_long['CELULAR']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)
df_long = df_long[
    (df_long['CELULAR'].notna()) &
    (df_long['CELULAR'] != '') &
    (df_long['CELULAR'].str.len() == 9) &
    (df_long['CELULAR'].str.startswith('9'))
]



In [3]:
filename='blacklist_celulares.txt'
filePath = os.path.join(ruta_csv, filename)

df_list = pd.read_csv(filePath)

In [4]:

print(f"df_dni filas: {df_long.shape[0]}")
print(f"df_list filas: {df_list.shape[0]}")
df_list['CELULAR'] = (
    df_list['CELULAR']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)

df_dni filas: 190898
df_list filas: 614450


In [26]:
dnis = [
    "70663492",
    "47002810",
    "75797316",
    "61273533",
    "10182850",
    "70884441",
    "07876887"
]

telefonos = [
    "983849201",
    "900927220",
    "986506300",
    "981422534",
    "989341699",
    "941800134",
    "998183675"
]

# import pandas as pd

df_manual_dni = pd.DataFrame({
    "dni_cliente": dnis,
})


df_manual_telf = pd.DataFrame({
    "CELULAR": telefonos
})


In [23]:
df_manual_telf.head()

,CELULAR
0,983849201
1,900927220
2,986506300
3,981422534
4,989341699


In [ ]:
data = [
    ("70663492", "983849201"),
    ("47002810", "900927220"),
    ("75797316", "986506300"),
    ("61273533", "981422534"),
    ("10182850", "989341699"),
    ("70884441", "941800134"),
    ("07876887", "998183675")
]


df_manual = pd.DataFrame(data, columns=["DNI", "CELULAR"])

In [21]:
spark_df = spark.createDataFrame(df_manual)
spark_df.show()


NameError: name 'spark' is not defined

In [25]:
df_manual.head()

,DNI,TELEFONO
0,70663492,983849201
1,47002810,900927220
2,75797316,986506300
3,61273533,981422534
4,10182850,989341699


In [27]:
df_manual_telf['CELULAR'] = df_manual_telf['CELULAR'].astype(str)

df_manual_telf = df_manual_telf.merge(
    df_long,
    on="CELULAR",
    how="inner"
)

print(f"df_list filas: {df_manual_telf.shape[0]}")


df_list filas: 0


In [6]:
df_list['cl_estado']='3'
df_list['estado']='Retirar Telf'
df_list=df_list[['NUMERO_DOCUMENTO','cl_estado','estado']]
df_list.count()

NUMERO_DOCUMENTO    417
cl_estado           417
estado              417
dtype: int64

In [7]:
df_list = df_list.drop_duplicates(
    subset=['NUMERO_DOCUMENTO'],
    keep='last'
)
df_list.count()

NUMERO_DOCUMENTO    216
cl_estado           216
estado              216
dtype: int64

In [10]:
df_list.head()

,col_01,col_03,col_02
1,45773218,3,Retirar Telf
3,43888745,3,Retirar Telf
5,04647692,3,Retirar Telf
7,47240319,3,Retirar Telf
9,71563325,3,Retirar Telf


In [9]:
reemplazo = {
    'NUMERO_DOCUMENTO': 'col_01',
    'estado': 'col_02',
    'cl_estado': 'col_03'
}
df_list = df_list.rename(columns=reemplazo)

In [10]:
from sqlalchemy import create_engine

engine = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)


In [11]:

df_list.to_sql(
    name="tb_temporal",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

216

In [12]:
from sqlalchemy import text

query = """
UPDATE crm_target.alfcc_clientes a
INNER JOIN crm_target.tb_temporal b
    ON a.NUMERO_DOCUMENTO = b.col_01 
SET 
    a.cl_estado = b.col_03,
    a.estado = b.col_02
WHERE 
    a.cl_base = 'mayo 2026'
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas afectadas:", result.rowcount)

Filas afectadas: 216


In [13]:
from sqlalchemy import text

query = """
DELETE FROM crm_target.tb_temporal
"""

with engine_mysql.begin() as conn:
    result = conn.execute(text(query))
    print("Filas eliminadas:", result.rowcount)

Filas eliminadas: 216


In [ ]:
from sqlalchemy import text

query = """
UPDATE crm_target.alfcc_clientes a
INNER JOIN crm_target.tb_temporal b
    ON a.NUMERO_DOCUMENTO COLLATE utf8mb4_general_ci
       = b.col_01 COLLATE utf8mb4_general_ci
SET 
    a.cl_estado = b.col_02,
    a.estado = b.col_03
WHERE 
    a.cl_base = 'mayo 2026'
    AND (
        IFNULL(a.cl_estado,'') COLLATE utf8mb4_general_ci 
            <> IFNULL(b.col_02,'') COLLATE utf8mb4_general_ci
        OR 
        IFNULL(a.estado,'') COLLATE utf8mb4_general_ci 
            <> IFNULL(b.col_03,'') COLLATE utf8mb4_general_ci
    )
LIMIT 1000
"""

with engine.begin() as conn:
    total = 0

    while True:
        result = conn.execute(text(query))
        filas = result.rowcount
        total += filas

        print("Filas afectadas:", filas)

        if filas == 0:
            break

    print("TOTAL ACTUALIZADO:", total)

In [ ]:

engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT 
* FROM tb_temporal
"""

df_prueba_01 = pd.read_sql(query, engine_mysql)

df_prueba_01.columns

Index(['col_01', 'col_02', 'col_03', 'col_04'], dtype='object')

In [8]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfcc_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="cl_estado",
    col_llave_df="NUMERO_DOCUMENTO",
    col_valor_df="retiro",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 11788
Lote 0 - 2000 actualizado | filas afectadas: 1927
Lote 2000 - 4000 actualizado | filas afectadas: 1931
Lote 4000 - 6000 actualizado | filas afectadas: 1908
Lote 6000 - 8000 actualizado | filas afectadas: 1898
Lote 8000 - 10000 actualizado | filas afectadas: 1897
Lote 10000 - 11788 actualizado | filas afectadas: 1690
Proceso terminado. Total filas afectadas: 11251


## actualizar retiro dni

In [28]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT 
DISTINCT 
NUMERO_DOCUMENTO as dni_cliente
FROM alfcc_clientes
WHERE cl_base = 'mayo 2026'
and cl_estado=1
"""

df_dni = pd.read_sql(query, engine_mysql)

df_dni["dni_cliente"] = (
    df_dni["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)


In [15]:
filename='blacklist_dni.txt'

filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)

In [30]:

# df_manual_dni.rename(columns={'DNI': 'dni_cliente'}, inplace=True)
df_manual_dni["dni_cliente"] = (
    df_manual_dni["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
print(df_manual_dni.columns)
print(df_dni.columns)


Index(['dni_cliente'], dtype='object')
Index(['dni_cliente'], dtype='object')


In [17]:
print(df_dni.columns)
print(df_list.columns)

Index(['dni_cliente'], dtype='object')
Index(['dni_cliente'], dtype='object')


NameError: name 'df_manual_dnil' is not defined

In [31]:
df_manual_dni = df_manual_dni.merge(
    df_dni,
    on="dni_cliente",
    how="inner"
)

print(f"df_list filas: {df_manual_dni.shape[0]}")


df_list filas: 0


In [19]:
df_list['retiro']='Retirar BlackList'
df_list=df_list[['dni_cliente','retiro']]
df_list.count()

dni_cliente    1648
retiro         1648
dtype: int64

In [ ]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfin_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="estado",
    col_llave_df="dni_cliente",
    col_valor_df="retiro",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 1648
Lote 0 - 1648 actualizado | filas afectadas: 0
Proceso terminado. Total filas afectadas: 0


## ejecutar query

In [ ]:
from sqlalchemy import create_engine


engine = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)
with engine.begin() as conn:
    result = conn.execute(text("""
        UPDATE crm_target.alfcc_clientes
        SET cl_estado = 3
        WHERE estado <> 'ACTIVO'
        AND cl_base = 'mayo 2026'
    """))
    
    print("Filas afectadas:", result.rowcount)

## actualizar dni a otro lote

In [3]:
filename='dni_repetidos_alfin.csv'

filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)

In [5]:
df_list["NUMERO_DOCUMENTO"] = (
    df_list["NUMERO_DOCUMENTO"]
    .astype(str)
    .str.zfill(8)
)
df_list.head()

,NUMERO_DOCUMENTO
0,46507674
1,40963267
2,41782588
3,40658062
4,09457398


In [6]:
df_list['lote_ref']='BD-Target -ASM'

In [7]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfin_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="lote",
    col_llave_df="NUMERO_DOCUMENTO",
    col_valor_df="lote_ref",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 1066
Lote 0 - 1066 actualizado | filas afectadas: 1066
Proceso terminado. Total filas afectadas: 1066


In [ ]:
BD-Target -ASM

In [5]:
exec_query_sql(server_zeus, "MAEBA", user_zeus, pwd_zeus, "ADM_OBJ_TG.spFunnelDinersTc", "SP funnel diners_tc Zeus")

SP funnel diners_tc Zeus | realizado | duración: 19.96 seg


In [ ]:
overwrite_table_SQL(spark,df_prueba_1,f'borrar_TARGET_202604_01',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba_2,f'borrar_TARGET_202604_02',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba_ch,f'borrar_TARGET_202604_ch',server_kishin,user_kishin,pwd_kishin,'DANTALION')


df_list filas: 1066


In [ ]:
df_dni filas: 128183
df_list filas: 12546

In [ ]:
ssss

In [7]:
print(df_list.columns)
print(df_long.columns)

Index(['TELEFONO'], dtype='object')
Index(['NUMERO_DOCUMENTO', 'tipo_telf', 'TELEFONO'], dtype='object')


df_list filas: 1


In [ ]:
df_list filas: 2036

NUMERO_DOCUMENTO    1
retiro              1
dtype: int64

In [10]:
df_list.head()

,NUMERO_DOCUMENTO,retiro
0,40503275,Retirar Telef


Total registros a procesar: 1
Lote 0 - 1 actualizado | filas afectadas: 1
Proceso terminado. Total filas afectadas: 1
